In [ ]:
import requests
from lxml import html
import pandas as pd
import time
import random
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

# recipe_f.csv 파일에서 recipe_id 목록 가져오기
recipe_ids_df = pd.read_csv('/Users/shindongeun/bigdata_lacture/team_project/mini3/recipe_f.csv')

# 당근, 양파, 돼지고기가 모두 포함된 레시피만 필터링
filtered_df = recipe_ids_df[recipe_ids_df['ingred'].apply(lambda x: '당근' in x and '양파' in x and '돼지고기' in x)]
recipe_ids = filtered_df['recipe_id'].unique()

# 모든 레시피 데이터를 저장할 리스트
all_recipe_data = []

# 요청; 크롬 드라이브 설정
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# 재시도 전략 설정
session = requests.Session()
retry = Retry(
    total=3,  # 최대 3번 재시도
    backoff_factor=1,  # 재시도 간격 (1초, 2초, 4초...)
    status_forcelist=[500, 502, 503, 504],  # 이 상태 코드일 때 재시도
)
adapter = HTTPAdapter(max_retries=retry)
session.mount('http://', adapter)
session.mount('https://', adapter)

# 진행 상황 추적
processed_count = 0
total_count = len(recipe_ids)

# 각 recipe_id에 대해 크롤링 수행
for recipe_id in recipe_ids:
    try:
        url = f"https://www.10000recipe.com/recipe/{recipe_id}"
        
        # 타임아웃 설정과 함께 요청
        response = session.get(url, headers=headers, timeout=10)
        response.encoding = 'utf-8'  # 한글 인코딩 설정
        
        # HTML 파싱
        tree = html.fromstring(response.content)
        
        # 대표 이미지 URL 추출
        food_img = tree.xpath('//*[@id="main_thumbs"]/@src')
        food_img = food_img[0] if food_img else None
        
        # 대표 이미지만 저장
        all_recipe_data.append({
            'recipe_id': recipe_id,
            'food_img': food_img
        })
        
        processed_count += 1
        print(f"✓ 처리 완료: {recipe_id} ({processed_count}/{total_count})")
        
        # 서버 부하를 줄이기 위한 랜덤 딜레이 (2~4초)
        time.sleep(random.uniform(2, 4))
        
        # 10개마다 중간 저장
        if processed_count % 10 == 0:
            temp_df = pd.DataFrame(all_recipe_data)
            temp_df.to_csv('all_recipes_images_temp.csv', index=False)
            print(f"💾 중간 저장 완료: {processed_count}개 처리됨")
        
    except requests.exceptions.Timeout:
        print(f"⏱ Timeout: {recipe_id} - 응답 시간 초과")
        continue
    except requests.exceptions.ConnectionError as e:
        print(f"🔌 Connection Error: {recipe_id} - {str(e)}")
        # 연결 에러 시 더 긴 대기 시간
        time.sleep(10)
        continue
    except Exception as e:
        print(f"❌ Error processing recipe {recipe_id}: {str(e)}")
        continue

# 전체 데이터프레임 생성
recipe_df = pd.DataFrame(all_recipe_data)

# CSV 파일로 저장
recipe_df.to_csv('all_food_images.csv', index=False)
print(f"\n✅ 완료! 총 {len(recipe_df)}개의 레시피 저장됨")
recipe_df

✓ 처리 완료: 398430 (1/1934)
✓ 처리 완료: 521359 (2/1934)
✓ 처리 완료: 587229 (3/1934)


KeyboardInterrupt: 

In [6]:
from IPython.display import Image, display, HTML

# HTML 테이블 형태로 이미지 표시
html_content = ""
for url in recipe_df['recipe_img_url']:
    if url:
        html_content += f'<img src="{url}" style="width:300px; margin:10px"/><br>'

display(HTML(html_content))